In [1]:
import subprocess
import glob
import platform

def run(command):
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
    )

    print(f"$ {command}")

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print(result.stderr)

    print("-" * 60)

print("Machine:", platform.machine())
print("Video devices:", glob.glob("/dev/video*"))
print()

run("gst-launch-1.0 --version")
run("gst-inspect-1.0 webrtcbin")
run("gst-inspect-1.0 v4l2src")
run("gst-inspect-1.0 vp8enc")
run("gst-inspect-1.0 x264enc")

Machine: armv7l
Video devices: ['/dev/video0', '/dev/video1']

$ gst-launch-1.0 --version
gst-launch-1.0 version 1.20.1
GStreamer 1.20.1
https://launchpad.net/distros/ubuntu/+source/gstreamer1.0

------------------------------------------------------------
$ gst-inspect-1.0 webrtcbin
Factory Details:
  Rank                     primary (256)
  Long-name                WebRTC Bin
  Klass                    Filter/Network/WebRTC
  Description              A bin for webrtc connections
  Author                   Matthew Waters <matthew@centricular.com>

Plugin Details:
  Name                     webrtc
  Description              WebRTC plugins
  Filename                 /usr/lib/arm-linux-gnueabihf/gstreamer-1.0/libgstwebrtc.so
  Version                  1.20.1
  License                  LGPL
  Source module            gst-plugins-bad
  Source release date      2022-03-14
  Binary package           GStreamer Bad Plugins (Ubuntu)
  Origin URL               https://launchpad.net/distros/ubunt

In [2]:
import subprocess

print(
    subprocess.run(
        ["cat", "/etc/os-release"],
        capture_output=True,
        text=True,
    ).stdout
)

NAME="PynqLinux"
VERSION="3.0 (Carlisle)"
ID=pynqlinux
ID_LIKE=ubuntu
PRETTY_NAME="PynqLinux, based on Ubuntu 22.04"
VERSION_ID="3.0"
HOME_URL="https://www.pynq.io/"
SUPPORT_URL="https://discuss.pynq.io/"
BUG_REPORT_URL="https://www.pynq.io"
PRIVACY_POLICY_URL="https://www.pynq.io"
VERSION_CODENAME=Carlisle
UBUNTU_CODENAME=jammy



In [3]:
import gi
import sys

gi.require_version("Gst", "1.0")
gi.require_version("GstWebRTC", "1.0")
gi.require_version("GstSdp", "1.0")

from gi.repository import Gst, GstWebRTC, GstSdp

Gst.init(None)

print("Python:", sys.version)
print("GStreamer:", Gst.version_string())
print("GstWebRTC: OK")
print("GstSdp: OK")

Python: 3.10.4 (main, Apr  2 2022, 09:04:19) [GCC 11.2.0]
GStreamer: GStreamer 1.20.1
GstWebRTC: OK
GstSdp: OK


In [4]:
try:
    import websockets
    print("websockets:", websockets.__version__)
except ImportError:
    print("websockets is not installed")

websockets: 10.3


In [5]:
import websockets

print("websockets version:", websockets.__version__)

websockets version: 10.3


In [6]:
import gi

gi.require_version("Gst", "1.0")
gi.require_version("GstWebRTC", "1.0")

from gi.repository import Gst, GstWebRTC

Gst.init(None)

PIPELINE_DESCRIPTION = """
webrtcbin name=webrtc bundle-policy=max-bundle

videotestsrc is-live=true pattern=ball
! videoconvert
! queue
! vp8enc deadline=1
! rtpvp8pay pt=96
! application/x-rtp,media=video,encoding-name=VP8,payload=96,clock-rate=90000
! webrtc.
"""

pipeline = Gst.parse_launch(PIPELINE_DESCRIPTION)

webrtc = pipeline.get_by_name("webrtc")

print("Pipeline created:", pipeline is not None)
print("webrtcbin found:", webrtc is not None)
print("webrtcbin:", webrtc)

Pipeline created: True
webrtcbin found: True
webrtcbin: <__gi__.GstWebRTCBin object at 0xae698648 (GstWebRTCBin at 0xfef170)>


In [7]:
import gi

gi.require_version("Gst", "1.0")
gi.require_version("GstWebRTC", "1.0")

from gi.repository import Gst, GstWebRTC

Gst.init(None)

elements = [
    "webrtcbin",
    "nicesrc",
    "nicesink",
    "dtlsenc",
    "dtlsdec",
    "srtpenc",
    "srtpdec",
    "rtpbin",
    "vp8enc",
    "rtpvp8pay",
]

for name in elements:
    factory = Gst.ElementFactory.find(name)
    print(f"{name:12}:", "OK" if factory else "MISSING")

webrtcbin   : OK
nicesrc     : OK
nicesink    : OK
dtlsenc     : OK
dtlsdec     : OK
srtpenc     : OK
srtpdec     : OK
rtpbin      : OK
vp8enc      : OK
rtpvp8pay   : OK


In [8]:
factory = Gst.ElementFactory.find("webrtcbin")

for template in factory.get_static_pad_templates():
    print(
        "name:", template.name_template,
        "| direction:", template.direction.value_nick,
        "| presence:", template.presence.value_nick,
        "| caps:", template.get_caps().to_string(),
    )

name: src_%u | direction: src | presence: sometimes | caps: application/x-rtp
name: sink_%u | direction: sink | presence: request | caps: application/x-rtp


In [9]:
import gi

gi.require_version("Gst", "1.0")
gi.require_version("GstWebRTC", "1.0")

from gi.repository import Gst, GstWebRTC

Gst.init(None)

pipeline = Gst.Pipeline.new("webrtc-test")

payloader = Gst.ElementFactory.make("rtpvp8pay", "payloader")
webrtc = Gst.ElementFactory.make("webrtcbin", "webrtc")

if pipeline is None:
    raise RuntimeError("Unable to create pipeline")

if payloader is None:
    raise RuntimeError("Unable to create rtpvp8pay")

if webrtc is None:
    raise RuntimeError("Unable to create webrtcbin")

pipeline.add(payloader)
pipeline.add(webrtc)

# Demande explicitement un request pad à webrtcbin.
sink_pad = webrtc.request_pad_simple("sink_%u")

print("Requested WebRTC pad:", sink_pad)

if sink_pad is None:
    raise RuntimeError("webrtcbin refused to create sink_%u")

src_pad = payloader.get_static_pad("src")

print("Payloader src pad:", src_pad)
print("WebRTC sink pad:", sink_pad)

print()
print("Payloader caps:")
print(src_pad.query_caps(None).to_string())

print()
print("WebRTC sink caps:")
print(sink_pad.query_caps(None).to_string())

print()

link_result = src_pad.link(sink_pad)

print("Pad link result:", link_result)
print("Pad link result name:", link_result.value_nick)

Requested WebRTC pad: <__gi__.GstWebRTCBinPad object at 0xae6a3a48 (GstWebRTCBinPad at 0x10271d8)>
Payloader src pad: <Gst.Pad object at 0xae6a3a88 (GstPad at 0x1002470)>
WebRTC sink pad: <__gi__.GstWebRTCBinPad object at 0xae6a3a48 (GstWebRTCBinPad at 0x10271d8)>

Payloader caps:
application/x-rtp, payload=(int)[ 96, 127 ], clock-rate=(int)90000, encoding-name=(string){ VP8, VP8-DRAFT-IETF-01 }

WebRTC sink caps:
application/x-rtp

Pad link result: <enum GST_PAD_LINK_OK of type Gst.PadLinkReturn>
Pad link result name: ok


In [10]:
pipeline = Gst.Pipeline.new("webrtc-h264-test")

payloader = Gst.ElementFactory.make("rtph264pay", "payloader")
webrtc = Gst.ElementFactory.make("webrtcbin", "webrtc")

if pipeline is None:
    raise RuntimeError("Unable to create pipeline")

if payloader is None:
    raise RuntimeError("Unable to create rtph264pay")

if webrtc is None:
    raise RuntimeError("Unable to create webrtcbin")

pipeline.add(payloader)
pipeline.add(webrtc)

sink_pad = webrtc.request_pad_simple("sink_%u")

if sink_pad is None:
    raise RuntimeError("Unable to request WebRTC sink pad")

src_pad = payloader.get_static_pad("src")

print("H264 RTP caps:")
print(src_pad.query_caps(None).to_string())

print()
print("WebRTC caps:")
print(sink_pad.query_caps(None).to_string())

result = src_pad.link(sink_pad)

print()
print("Link result:", result.value_nick)

H264 RTP caps:
application/x-rtp, media=(string)video, payload=(int)[ 96, 127 ], clock-rate=(int)90000, encoding-name=(string)H264

WebRTC caps:
application/x-rtp

Link result: ok


In [11]:
import gi

gi.require_version("Gst", "1.0")
gi.require_version("GstWebRTC", "1.0")
gi.require_version("GstSdp", "1.0")

from gi.repository import Gst, GstWebRTC, GstSdp

Gst.init(None)

# On laisse volontairement webrtcbin hors du parse_launch,
# puisque la liaison automatique vers son request pad pose problème.
PIPELINE_DESCRIPTION = """
v4l2src device=/dev/video0
! image/jpeg,width=640,height=480,framerate=30/1
! jpegdec
! videorate
! video/x-raw,framerate=15/1
! videoconvert
! video/x-raw,format=I420
! queue max-size-buffers=2 leaky=downstream
! x264enc
    tune=zerolatency
    speed-preset=ultrafast
    bitrate=800
    key-int-max=30
! video/x-h264,profile=constrained-baseline
! h264parse
! rtph264pay
    name=payloader
    pt=96
    config-interval=-1
    aggregate-mode=zero-latency
"""

pipeline = Gst.parse_launch(PIPELINE_DESCRIPTION)

payloader = pipeline.get_by_name("payloader")

if payloader is None:
    raise RuntimeError("Unable to find RTP payloader")

# Création séparée de webrtcbin
webrtc = Gst.ElementFactory.make("webrtcbin", "webrtc")

if webrtc is None:
    raise RuntimeError("Unable to create webrtcbin")


def on_connection_state_changed(element, _):
    state = element.get_property("connection-state")
    print("PYNQ WebRTC connection state:", state.value_nick)


def on_ice_connection_state_changed(element, _):
    state = element.get_property("ice-connection-state")
    print("PYNQ ICE connection state:", state.value_nick)


def on_ice_gathering_state_changed(element, _):
    state = element.get_property("ice-gathering-state")
    print("PYNQ ICE gathering state:", state.value_nick)


webrtc.connect(
    "notify::connection-state",
    on_connection_state_changed,
)

webrtc.connect(
    "notify::ice-connection-state",
    on_ice_connection_state_changed,
)

webrtc.connect(
    "notify::ice-gathering-state",
    on_ice_gathering_state_changed,
)

pipeline.add(webrtc)

# Request pad manuel
webrtc_sink_pad = webrtc.request_pad_simple("sink_%u")

if webrtc_sink_pad is None:
    raise RuntimeError("Unable to request WebRTC sink pad")

rtp_src_pad = payloader.get_static_pad("src")

if rtp_src_pad is None:
    raise RuntimeError("Unable to get RTP source pad")

link_result = rtp_src_pad.link(webrtc_sink_pad)

if link_result != Gst.PadLinkReturn.OK:
    raise RuntimeError(
        f"Unable to link RTP to WebRTC: {link_result.value_nick}"
    )

print("Camera pipeline created: OK")
print("WebRTC element created: OK")
print("WebRTC sink pad:", webrtc_sink_pad.get_name())
print("RTP -> WebRTC:", link_result.value_nick)

Camera pipeline created: OK
WebRTC element created: OK
WebRTC sink pad: sink_0
RTP -> WebRTC: ok


In [12]:
import asyncio
import json
import websockets

clients = set()
event_loop = asyncio.get_running_loop()


async def broadcast(message):
    if not clients:
        return

    dead = []

    for websocket in clients:
        try:
            await websocket.send(message)
        except Exception:
            dead.append(websocket)

    for websocket in dead:
        clients.discard(websocket)


def send_json_from_gstreamer(payload):
    message = json.dumps(payload)

    asyncio.run_coroutine_threadsafe(
        broadcast(message),
        event_loop,
    )


def on_ice_candidate(element, mline_index, candidate):
    print(
        f"PYNQ local ICE: "
        f"mline={mline_index} candidate={candidate}"
    )

    send_json_from_gstreamer({
        "type": "ice",
        "candidate": candidate,
        "sdpMLineIndex": mline_index,
    })


def on_offer_created(promise, _, __):
    promise.wait()

    reply = promise.get_reply()
    offer = reply.get_value("offer")

    webrtc.emit("set-local-description", offer, None)

    sdp_text = offer.sdp.as_text()

    print("SDP offer created")

    send_json_from_gstreamer({
        "type": "offer",
        "sdp": sdp_text,
    })


def on_negotiation_needed(element):
    print("Negotiation needed")

    promise = Gst.Promise.new_with_change_func(
        on_offer_created,
        None,
        None,
    )

    element.emit("create-offer", None, promise)


webrtc.connect(
    "on-negotiation-needed",
    on_negotiation_needed,
)

webrtc.connect(
    "on-ice-candidate",
    on_ice_candidate,
)

print("WebRTC callbacks configured")

WebRTC callbacks configured


In [13]:
pipeline_started = False

from websockets.exceptions import ConnectionClosed

async def handle_client(websocket, path):
    global pipeline_started

    print("Flutter client connected")
    clients.add(websocket)

    try:
        if not pipeline_started:
            state = pipeline.set_state(Gst.State.PLAYING)
            print("Pipeline state:", state.value_nick)
            pipeline_started = True

        async for raw_message in websocket:
            message = json.loads(raw_message)

            print("Received:", message.get("type"))

            if message["type"] == "answer":
                sdp_text = message["sdp"]

                _, sdp = GstSdp.SDPMessage.new()

                result = GstSdp.sdp_message_parse_buffer(
                    sdp_text.encode(),
                    sdp,
                )

                if result != GstSdp.SDPResult.OK:
                    raise RuntimeError("Unable to parse SDP answer")

                answer = GstWebRTC.WebRTCSessionDescription.new(
                    GstWebRTC.WebRTCSDPType.ANSWER,
                    sdp,
                )

                promise = Gst.Promise.new()

                webrtc.emit(
                    "set-remote-description",
                    answer,
                    promise,
                )

                promise.interrupt()

                print("Remote SDP answer applied")

            elif message["type"] == "ice":
                print(
                    "Flutter ICE received:",
                    message.get("candidate"),
                )

                webrtc.emit(
                    "add-ice-candidate",
                    message["sdpMLineIndex"],
                    message["candidate"],
                )

                print("Remote ICE candidate added")

    except ConnectionClosed:
        print("Flutter WebSocket closed")

    finally:
        clients.discard(websocket)
        print("Flutter client disconnected")

In [14]:
signaling_server = await websockets.serve(
    handle_client,
    "0.0.0.0",
    8765,
)

print("Signaling server running on ws://0.0.0.0:8765")

Signaling server running on ws://0.0.0.0:8765


Flutter client connected
Pipeline state: success
Negotiation needed
SDP offer created
PYNQ ICE gathering state: gathering
PYNQ local ICE: mline=0 candidate=candidate:1 1 UDP 2015363327 192.168.2.99 50899 typ host
PYNQ local ICE: mline=0 candidate=candidate:2 1 TCP 1015021823 192.168.2.99 9 typ host tcptype active
PYNQ local ICE: mline=0 candidate=candidate:3 1 TCP 1010827519 192.168.2.99 60661 typ host tcptype passive
PYNQ local ICE: mline=0 candidate=candidate:4 1 UDP 2015363583 192.168.200.111 39425 typ host
PYNQ local ICE: mline=0 candidate=candidate:5 1 TCP 1015022079 192.168.200.111 9 typ host tcptype active
PYNQ local ICE: mline=0 candidate=candidate:6 1 TCP 1010827775 192.168.200.111 56841 typ host tcptype passive
PYNQ local ICE: mline=0 candidate=candidate:7 1 UDP 2015363839 fe80::3440:20ff:fe6c:9b78 56878 typ host
PYNQ local ICE: mline=0 candidate=candidate:8 1 TCP 1015022335 fe80::3440:20ff:fe6c:9b78 9 typ host tcptype active
PYNQ local ICE: mline=0 candidate=candidate:9 1 TC